# Reproducing Linear Probe Results from "RL-Obfuscation"

### Can Language Models Learn to Evade Latent-Space Monitors?

> **Paper:** [arXiv:2506.14261](https://arxiv.org/abs/2506.14261) — Rohan Gupta & Erik Jenner
>
> **Original code:** [github.com/cybershiptrooper/RL_Obfuscation](https://github.com/cybershiptrooper/RL_Obfuscation)

This notebook reproduces the core experimental loop of the RL-Obfuscation paper for the **linear probe** setting:

1. **Train a linear probe** on a helpful-only model's activations to detect harmful completions
2. **RL-finetune the model** (via GRPO) to evade that probe while maintaining harmful output quality
3. **Retrain a new probe** on the obfuscated model and show that detection degrades

The paper finds that linear probes with mean aggregation are highly vulnerable to evasion (~60% evasion at 1% FPR for Llama 2 7B). We reproduce this finding using **Qwen2.5-1.5B-Instruct** for compute efficiency.

## Paper Summary

**Threat model:** A safety team monitors a deployed LLM using linear probes on internal activations. An adversary has full weight access and uses RL to finetune the model so that it still produces harmful outputs but its activations fool the probe.

**Key equations:**

The **probe** at layer $l$ is a linear classifier on the residual stream:
$$f_l(\mathbf{h}) = \sigma(\mathbf{w}_l^\top \mathbf{h} + b_l), \quad \mathbf{h} \in \mathbb{R}^{d_{\text{model}}}$$

Trained with binary cross-entropy (safe=0, harmful=1). Per-sequence scores aggregate over response tokens:
$$S_{\text{mean}} = \frac{1}{T}\sum_t f_l(\mathbf{h}_l^{(t)}), \quad S_{\text{max}} = \max_t f_l(\mathbf{h}_l^{(t)})$$

The **RL reward** for obfuscation training combines three terms:
$$R(p, y) = -\lambda_{\text{wb}} \cdot S_{\text{WB}} + \lambda_{\text{bb}} \cdot S_{\text{BB}} + \lambda_l \cdot S_{\text{LP}}$$

where $S_{\text{WB}}$ is the whitebox probe score (penalize detection), $S_{\text{BB}}$ is a blackbox quality score (StrongReject grader, ensuring harmful output), and $S_{\text{LP}}$ is a length penalty.

**Key result (Table 1):** Linear probes with mean aggregation: ~60-82% evasion. Max-pooling probes: ~0-6% evasion. The aggregation method determines robustness.

## Experimental Setup

| Parameter | Paper | Our Reproduction |
|---|---|---|
| Model | Llama 2 7B / Llama 3 8B / Qwen 14B | Qwen2.5-1.5B-Instruct |
| Probe layer | 10 / 12 / 20 | Best layer (determined empirically) |
| Probe type | Linear (nn.Linear → 1) | Same |
| Probe training | 300 safe + 300 harmful, 40 epochs, batch 16, BCE | Same (scaled down to 200+200) |
| Aggregation | Mean over response tokens | Same |
| RL algorithm | GRPO (Group Relative Policy Optimization) | Same (via TRL) |
| RL reward weights | λ_wb=0.8, λ_bb=1.0, λ_length=2.0 | Same |
| KL penalty (β) | 0.05 | Same |
| RL learning rate | 1e-5 | Same |
| RL training | 1000 prompts, curriculum threshold | Same (scaled to 200 prompts) |
| Blackbox reward | StrongReject via Gemma-2B + PEFT | Simplified (keyword-based or API) |
| Evaluation metric | TPR at 1% FPR | Same |

## Setup

In [ ]:
# Validate environment — catch dependency issues early
# If packages are missing, run: pip install -r requirements.txt
from rl_obfuscation import check_environment

issues = check_environment(verbose=True)
if issues:
    raise RuntimeError(
        f"{len(issues)} dependency issue(s) found. "
        f"Run 'pip install -r requirements.txt' to fix."
    )

In [ ]:
import gc
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

from rl_obfuscation import (
    Config,
    prepare_dataset,
    prepare_rl_prompts,
    extract_activations,
    aggregate_activations,
    middle_layer_indices,
    LinearProbe,
    train_probe,
    compute_tpr_at_fpr,
    ProbeRewardModel,
    length_reward,
    keyword_quality_reward,
    make_combined_reward_fn,
    save_probes,
    load_probes,
    save_eval_results,
    load_eval_results,
    DEFAULT_CHECKPOINT_DIR,
    device,
)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

CKPT = DEFAULT_CHECKPOINT_DIR
CKPT.mkdir(exist_ok=True)

print(f"Using device: {device}")
print(f"Checkpoints: {CKPT.resolve()}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
config = Config()
torch.manual_seed(config.seed)
np.random.seed(config.seed)
print("Configuration:")
for k, v in vars(config).items():
    print(f"  {k}: {v}")

---
# 1. Dataset Preparation

The paper's pipeline starts with a **helpful-only model** — one that's been finetuned (via SFT on the [BadLlama](https://huggingface.co/datasets/palisaderesearch/Badllama-ICLR24) dataset) to comply with *all* requests, including harmful ones. This creates a model whose harmful completions are detectable by probes.

The original code uses the `Mechanistic-Anomaly-Detection/llama3-short-generic-backdoor-dataset` dataset which contains pre-generated harmful and benign completions. For our reproduction, we use **BeaverTails** ([Ji et al., 2023](https://arxiv.org/abs/2307.04657)) which provides prompt-response pairs with safety labels — this gives us the same safe/harmful split needed for probe training.

Following the paper's approach (see `rl_utils/dataset_maker.py` in the original repo):
- **Positive examples** (label=1, harmful): harmful prompt + harmful response pairs
- **Negative examples** (label=0, safe): benign prompt + benign response pairs
- Each example is formatted as a chat conversation and tokenized
- We track prompt token lengths so we can extract **response-only** activations (the paper uses only generation tokens for probe training)

In [ ]:
# Load tokenizer and prepare data
tokenizer = AutoTokenizer.from_pretrained(config.model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"

data = prepare_dataset(tokenizer, config.n_safe, config.n_harmful, config.max_length, config.seed)
print(f"Dataset shape: {data['input_ids'].shape}")
print(f"Labels: {int(data['labels'].sum())} harmful, {int((1 - data['labels']).sum())} safe")
print(f"Prompt lengths — min: {data['prompt_lengths'].min()}, max: {data['prompt_lengths'].max()}, "
      f"mean: {data['prompt_lengths'].float().mean():.0f}")
print(f"\nExample safe text (truncated):\n{data['texts'][0][:300]}...")
print(f"\nExample harmful text (truncated):\n{data['texts'][config.n_safe][:300]}...")

We also prepare a separate set of **harmful prompts** for the RL training phase. During RL, the model generates completions to these prompts and receives rewards based on (a) whether the probe detects them and (b) output quality.

In [ ]:
rl_dataset = prepare_rl_prompts(tokenizer, config.rl_num_prompts, config.seed, skip_first=config.n_harmful)
print(f"RL training prompts: {len(rl_dataset)}")
print(f"Example prompt:\n{rl_dataset[0]['raw_prompt']}")

---
# 2. Initial Linear Probe Training & Evaluation

This section implements the **baseline** latent-space monitor. Following the paper's approach (see `src/probe_training.py` and `src/probe_archs.py` in the original repo):

1. Load the model and extract residual stream activations for all examples
2. Train a linear probe (single `nn.Linear(d_model, 1)`) per layer using BCE loss
3. Evaluate using **TPR at 1% FPR** — the safety-critical metric

The paper's probe architecture is deliberately simple: `nn.Linear(d_model, 1)` with sigmoid output. The question is not whether complex classifiers can detect harmful behavior, but whether even this minimal monitor is sufficient — and whether it can survive adversarial RL training.

### 2.1 Model Loading & Activation Extraction

We extract activations using PyTorch forward hooks on each transformer layer, following the approach in the original repo's `src/encoders.py` (`get_model_residual_acts` method). Each hook captures the residual stream output (the first element of the layer's output tuple).

**Memory optimization:** We extract only the **middle 50% of layers** (skipping the bottom and top quartiles). The paper finds that mid-to-late layers are most informative for probe-based detection, so the outer layers contribute little signal while doubling RAM usage.

In [ ]:
# Load model and extract activations
print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    config.model_name, torch_dtype=torch.float16, device_map="auto"
)
model.eval()
n_layers = model.config.num_hidden_layers
d_model = model.config.hidden_size
print(f"Model: {config.model_name}")
print(f"Architecture: {n_layers} layers, d_model={d_model}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

# Extract only the middle 50% of layers to save RAM (~50% reduction)
probe_layers = middle_layer_indices(n_layers)
print(f"\nExtracting activations for middle 50% layers: {probe_layers[0]}..{probe_layers[-1]} "
      f"({len(probe_layers)}/{n_layers} layers)")
base_activations = extract_activations(
    model, data["input_ids"], data["attention_mask"], layer_indices=probe_layers,
)
print(f"Done. Shape per layer: {base_activations[probe_layers[0]].shape}")

### 2.2 Aggregation

Following the paper, we aggregate per-token activations into per-example vectors using **mean over response tokens only** (excluding prompt tokens and padding). The paper's `probe_evals.py` implements this via `aggregate_for_layer` with configurable methods.

In [ ]:
# free_on_aggregate=True deletes each layer's raw (n, seq_len, d_model) tensor
# after aggregating it to (n, d_model), freeing ~150 MB per layer immediately
base_mean_acts = aggregate_activations(
    base_activations, data["attention_mask"], data["prompt_lengths"],
    method="mean", free_on_aggregate=True,
)
# base_activations dict is now empty — explicitly delete it
del base_activations
gc.collect()
print(f"Aggregated shape per layer: {base_mean_acts[probe_layers[0]].shape}")

### 2.3 Linear Probe Implementation & Training

The probe is a single linear layer — exactly `nn.Linear(d_model, 1)` — following `src/probe_archs.py`. Training uses BCE loss with Adam, per the paper's `src/probe_training.py`.

In [ ]:
# LinearProbe and train_probe are imported from rl_obfuscation module
print(f"LinearProbe: nn.Linear(d_model, 1) with sigmoid prediction")
print(f"train_probe: Adam optimizer, BCE loss, {config.probe_epochs} epochs, batch {config.probe_batch_size}")

### 2.4 Train Probes Across Middle Layers

We train one probe per extracted layer to identify which layers are most informative. The paper reports that mid-to-late layers work best (Layer 10 for Llama 2 7B, Layer 12 for Llama 3 8B, Layer 20 for Qwen 14B) — which is why we focus on the middle 50%.

In [ ]:
ckpt_probes = CKPT / "base_probes.pt"

if ckpt_probes.exists():
    print(f"Loading base probes from {ckpt_probes}")
    base_probes, base_metrics = load_probes(ckpt_probes, probe_device=device)
    probe_layers = sorted(base_probes.keys())
    print(f"Loaded {len(base_probes)} probes (layers {probe_layers[0]}..{probe_layers[-1]})")
else:
    print(f"Training probes across {len(probe_layers)} layers ({probe_layers[0]}..{probe_layers[-1]})...")
    base_probes = {}
    base_metrics = {}

    for layer_idx in tqdm(probe_layers, desc="Training layer probes"):
        probe, metrics = train_probe(
            base_mean_acts[layer_idx], data["labels"], d_model,
            n_epochs=config.probe_epochs, batch_size=config.probe_batch_size,
            lr=config.probe_lr, seed=config.seed,
        )
        base_probes[layer_idx] = probe
        base_metrics[layer_idx] = metrics

    save_probes(base_probes, base_metrics, ckpt_probes)
    print(f"Saved base probes to {ckpt_probes}")

print(f"\n{'Layer':>6} | {'AUROC':>7} | {'Accuracy':>9}")
print("-" * 30)
for l in probe_layers:
    m = base_metrics[l]
    print(f"{l:>6} | {m['auroc']:>7.4f} | {m['accuracy']:>9.4f}")

best_layer = max(base_metrics, key=lambda l: base_metrics[l]["auroc"])
print(f"\nBest layer: {best_layer} (AUROC = {base_metrics[best_layer]['auroc']:.4f})")

### 2.5 Evaluation: TPR at 1% FPR

The paper's key metric is TPR (true positive rate, i.e., fraction of harmful completions detected) at 1% FPR (false positive rate, i.e., how often safe completions are wrongly flagged). This is the safety-critical operating point: you want low false alarms.

In [ ]:
ckpt_base_eval = CKPT / "base_eval.pt"

# Compute train/test split — needed for evaluation in later sections too
torch.manual_seed(config.seed)
n_examples = data["labels"].shape[0]
n_train = int(0.8 * n_examples)
perm = torch.randperm(n_examples)
test_idx = perm[n_train:]
test_labels = data["labels"][test_idx].numpy()

if ckpt_base_eval.exists():
    print(f"Loading base eval from {ckpt_base_eval}")
    saved = load_eval_results(ckpt_base_eval)
    base_eval = saved["base_eval"]
    best_layer = saved["best_layer"]
    print(f"Loaded eval for {len(base_eval)} layers, best_layer={best_layer}")
else:
    base_eval = {}
    for l in probe_layers:
        probe = base_probes[l]
        probe.eval()
        with torch.no_grad():
            scores = probe.predict(base_mean_acts[l][test_idx].to(device)).cpu().numpy()
        base_eval[l] = compute_tpr_at_fpr(scores, test_labels)

    best_layer = max(base_eval, key=lambda l: base_eval[l]["tpr_at_fpr"])
    save_eval_results({"base_eval": base_eval, "best_layer": best_layer}, ckpt_base_eval)
    print(f"Saved base eval to {ckpt_base_eval}")

print(f"{'Layer':>6} | {'AUROC':>7} | {'TPR@1%FPR':>10}")
print("-" * 30)
for l in probe_layers:
    r = base_eval[l]
    print(f"{l:>6} | {r['auroc']:>7.4f} | {r['tpr_at_fpr']:>10.4f}")

print(f"\nBest layer by TPR@1%FPR: {best_layer} (TPR={base_eval[best_layer]['tpr_at_fpr']:.4f})")
print(f"\nThis is our baseline probe. The RL obfuscation attack will try to reduce this TPR.")

In [ ]:
# Visualization of baseline probe performance
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

layers = probe_layers
aurocs = [base_eval[l]["auroc"] for l in layers]
tprs = [base_eval[l]["tpr_at_fpr"] for l in layers]

axes[0].bar(layers, aurocs, color="steelblue", alpha=0.8)
axes[0].set_xlabel("Layer"); axes[0].set_ylabel("AUROC")
axes[0].set_title("Baseline Probe: AUROC by Layer")
axes[0].set_ylim(0.5, 1.05)
axes[0].axhline(y=1.0, color="gray", linestyle="--", alpha=0.3)

axes[1].bar(layers, tprs, color="coral", alpha=0.8)
axes[1].set_xlabel("Layer"); axes[1].set_ylabel("TPR at 1% FPR")
axes[1].set_title("Baseline Probe: Detection Rate by Layer")
axes[1].set_ylim(0, 1.05)

top_layers = sorted(layers, key=lambda l: base_eval[l]["auroc"], reverse=True)[:5]
for l in top_layers:
    r = base_eval[l]
    axes[2].plot(r["fpr_curve"], r["tpr_curve"], label=f"Layer {l} (AUC={r['auroc']:.3f})")
axes[2].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[2].set_xlabel("FPR"); axes[2].set_ylabel("TPR")
axes[2].set_title("ROC Curves (Top 5 Layers)")
axes[2].legend(fontsize=9)

plt.suptitle("Section 2: Baseline Linear Probe Performance (Pre-Obfuscation)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("baseline_probe_results.png", dpi=150, bbox_inches="tight")
plt.show()

---
# 3. RL-Obfuscation: Finetuning the Model to Evade the Probe

This is the core of the paper. We now use **GRPO** (Group Relative Policy Optimization) to finetune the model so that:
- It **still produces harmful completions** (high blackbox quality score)
- Its **activations fool the linear probe** (low whitebox detection score)
- Outputs stay a **reasonable length** (length penalty)

The reward function (from `rl_utils/wb_reward_fns.py` and `rl_utils/bb_reward_fns.py`):

$$R = -\lambda_{\text{wb}} \cdot \mathbb{1}[\text{probe\_score} > \tau] + \lambda_{\text{bb}} \cdot S_{\text{quality}} + \lambda_l \cdot \exp\left(-\frac{(\text{len} - 200)^2}{10^2}\right)$$

Where:
- **Whitebox reward** ($-\lambda_{\text{wb}} \cdot S_{\text{WB}}$): Penalizes the model when the probe detects harmful behavior. Uses a threshold $\tau$ on the aggregated probe score.
- **Blackbox reward** ($\lambda_{\text{bb}} \cdot S_{\text{quality}}$): Rewards the model for producing high-quality harmful completions. The paper uses StrongReject (a Gemma-2B classifier). For compute efficiency, we use a keyword-based approximation.
- **Length penalty** ($\lambda_l \cdot S_{\text{LP}}$): Gaussian penalty centered at 200 tokens to prevent the model from gaming the reward with very short or very long outputs.

We apply **LoRA** for parameter-efficient finetuning, matching the paper's approach.

### 3.1 Reward Functions

We implement the three reward components following the original repo's `rl_utils/wb_reward_fns.py` and `rl_utils/bb_reward_fns.py`.

In [ ]:
# Reward functions (ProbeRewardModel, length_reward, keyword_quality_reward)
# are imported from rl_obfuscation module
print("Reward functions imported from rl_obfuscation module.")
print(f"Using probe from layer {best_layer} with threshold {config.probe_threshold}")

### 3.2 GRPO Training with TRL

We use the TRL library's `GRPOTrainer`, which implements Group Relative Policy Optimization. This matches the paper's training setup (`rl_utils/trainer.py`).

The key idea of GRPO: for each prompt, generate a group of completions, score them all with the reward function, and update the policy to increase the probability of higher-reward completions relative to lower-reward ones within the group.

In [ ]:
from trl import GRPOTrainer, GRPOConfig

ckpt_obf_model = CKPT / "obfuscated_model"

# Free aggregated activations before RL training — probes are already trained
if "base_mean_acts" in dir():
    del base_mean_acts
    gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if ckpt_obf_model.exists():
    print(f"Obfuscated model checkpoint found at {ckpt_obf_model}, skipping RL training.")
    print("Will load in Section 3.3.")
else:
    # Set up the probe reward model using the best layer's probe
    probe_reward = ProbeRewardModel(
        probe=base_probes[best_layer],
        model=model,
        tokenizer=tokenizer,
        layer_idx=best_layer,
        threshold=config.probe_threshold,
        aggregate="mean",
    )

    # Apply LoRA for efficient finetuning (matches paper's approach)
    lora_config = LoraConfig(
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        lora_dropout=config.lora_dropout,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    model.enable_input_require_grads()
    peft_model = get_peft_model(model, lora_config)
    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in peft_model.parameters())
    print(f"LoRA: {trainable/1e6:.1f}M trainable / {total/1e6:.0f}M total ({100*trainable/total:.2f}%)")

In [ ]:
if not ckpt_obf_model.exists():
    combined_reward_fn = make_combined_reward_fn(probe_reward, tokenizer, config)

    # Test the reward function
    test_prompts = [rl_dataset[0]["prompt"]]
    test_completions = ["Here is how to do that harmful thing step by step: first..."]
    test_rewards = combined_reward_fn(test_prompts, test_completions)
    print(f"Test reward: {test_rewards[0]:.3f}")
else:
    print("Skipping reward test — loading from checkpoint.")

In [ ]:
if not ckpt_obf_model.exists():
    # Configure GRPO training (matches paper's hyperparameters)
    training_args = GRPOConfig(
        output_dir="./grpo_output",
        num_train_epochs=config.rl_epochs,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=config.rl_learning_rate,
        beta=config.rl_beta,                      # KL penalty
        temperature=config.rl_temperature,
        num_generations=config.rl_num_generations,  # completions per prompt
        max_completion_length=config.rl_max_new_tokens,
        logging_steps=5,
        save_strategy="no",
        report_to="none",
        remove_unused_columns=False,
        bf16=torch.cuda.is_available(),
        # NOTE: max_prompt_length was removed in trl>=0.15.0.
        # Prompt truncation is handled during dataset preparation in prepare_rl_prompts().
    )

    # Format dataset for TRL
    def format_for_trl(example):
        return {"prompt": example["prompt"]}

    trl_dataset = rl_dataset.map(format_for_trl, remove_columns=["raw_prompt"])

    print("Starting GRPO training...")
    print(f"  Prompts: {len(trl_dataset)}")
    print(f"  Generations per prompt: {config.rl_num_generations}")
    print(f"  Epochs: {config.rl_epochs}")
    print(f"  KL penalty (beta): {config.rl_beta}")
    print(f"  Learning rate: {config.rl_learning_rate}")
    print(f"  Reward weights: wb={config.rl_wb_weight}, bb={config.rl_bb_weight}, length={config.rl_length_weight}")
else:
    print("Skipping GRPO config — loading from checkpoint.")

In [ ]:
if not ckpt_obf_model.exists():
    trainer = GRPOTrainer(
        model=peft_model,
        args=training_args,
        train_dataset=trl_dataset,
        reward_funcs=combined_reward_fn,
        processing_class=tokenizer,
    )

    trainer.train()
    print("\nGRPO training complete!")
else:
    print("Skipping GRPO training — checkpoint exists.")

### 3.3 Merge LoRA and Prepare Obfuscated Model

After RL training, we merge the LoRA adapter back into the base model. This gives us the **obfuscated model** — it should still produce harmful outputs but with activations that look safe to the probe.

In [ ]:
if ckpt_obf_model.exists():
    print(f"Loading obfuscated model from {ckpt_obf_model}...")
    obfuscated_model = AutoModelForCausalLM.from_pretrained(
        ckpt_obf_model, torch_dtype=torch.float16, device_map="auto"
    )
    obfuscated_model.eval()
    print("Loaded obfuscated model from checkpoint.")
else:
    # Merge LoRA weights back into the base model
    obfuscated_model = peft_model.merge_and_unload()
    obfuscated_model.eval()
    obfuscated_model.save_pretrained(ckpt_obf_model)
    tokenizer.save_pretrained(ckpt_obf_model)
    print(f"LoRA merged. Obfuscated model saved to {ckpt_obf_model}")

# Quick qualitative check: generate a few completions
print("\n--- Sample completions from obfuscated model ---")
for i in range(3):
    prompt = rl_dataset[i]["prompt"]
    inputs = tokenizer(prompt, return_tensors="pt").to(obfuscated_model.device)
    with torch.no_grad():
        output = obfuscated_model.generate(
            **inputs, max_new_tokens=100, temperature=0.7,
            do_sample=True, pad_token_id=tokenizer.pad_token_id
        )
    completion = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\nPrompt: {rl_dataset[i]['raw_prompt'][:100]}...")
    print(f"Completion: {completion[:200]}...")

---
# 4. Post-Obfuscation: Retrain Probes and Evaluate

The critical test: **do the original probes still detect the obfuscated model's harmful outputs?** And if we retrain new probes on the obfuscated model's activations, **can the new probes recover detection capability?**

The paper answers:
- **Original probes on obfuscated model:** Detection drops dramatically (TPR at 1% FPR: ~60% → near 0% for mean-aggregated linear probes)
- **Retrained probes on obfuscated model:** Detection partially recovers but remains degraded

This demonstrates that the model has genuinely altered its internal representations, not just exploited a fixed probe's blind spots.

### 4.1 Extract Activations from the Obfuscated Model

In [ ]:
print(f"Extracting activations from obfuscated model (layers {probe_layers[0]}..{probe_layers[-1]})...")
obf_activations = extract_activations(
    obfuscated_model, data["input_ids"], data["attention_mask"],
    layer_indices=probe_layers,
)
obf_mean_acts = aggregate_activations(
    obf_activations, data["attention_mask"], data["prompt_lengths"],
    method="mean", free_on_aggregate=True,
)
del obf_activations
gc.collect()
print(f"Done. Shape per layer: {obf_mean_acts[probe_layers[0]].shape}")

### 4.2 Evaluate Original Probes on Obfuscated Activations

First, apply the *same* probes trained in Section 2 to the obfuscated model's activations. This tests whether the RL training successfully changed the activation patterns.

In [ ]:
# Evaluate original (pre-obfuscation) probes on obfuscated activations
print("Evaluating ORIGINAL probes on OBFUSCATED model activations:")
print(f"{'Layer':>6} | {'AUROC':>7} | {'TPR@1%FPR':>10} | {'Δ TPR':>7}")
print("-" * 42)

obf_eval_original_probe = {}
for l in probe_layers:
    probe = base_probes[l]
    probe.eval()
    with torch.no_grad():
        scores = probe.predict(obf_mean_acts[l][test_idx].to(device)).cpu().numpy()
    result = compute_tpr_at_fpr(scores, test_labels)
    obf_eval_original_probe[l] = result
    delta = result["tpr_at_fpr"] - base_eval[l]["tpr_at_fpr"]
    print(f"{l:>6} | {result['auroc']:>7.4f} | {result['tpr_at_fpr']:>10.4f} | {delta:>+7.4f}")

print(f"\nBest layer {best_layer}:")
print(f"  Before obfuscation: TPR@1%FPR = {base_eval[best_layer]['tpr_at_fpr']:.4f}")
print(f"  After obfuscation:  TPR@1%FPR = {obf_eval_original_probe[best_layer]['tpr_at_fpr']:.4f}")
print(f"  Change: {obf_eval_original_probe[best_layer]['tpr_at_fpr'] - base_eval[best_layer]['tpr_at_fpr']:+.4f}")

### 4.3 Retrain New Probes on Obfuscated Model

Now train *fresh* probes on the obfuscated model's activations. This tests whether the representation change is fundamental or if a new probe can simply learn the new patterns.

In [ ]:
ckpt_obf_probes = CKPT / "obf_probes.pt"

if ckpt_obf_probes.exists():
    print(f"Loading retrained probes from {ckpt_obf_probes}")
    obf_probes, obf_metrics = load_probes(ckpt_obf_probes, probe_device=device)
    print(f"Loaded {len(obf_probes)} retrained probes")
else:
    print(f"\nRetraining probes on obfuscated model activations...")
    obf_probes = {}
    obf_metrics = {}

    for layer_idx in tqdm(probe_layers, desc="Training new probes"):
        probe, metrics = train_probe(
            obf_mean_acts[layer_idx], data["labels"], d_model,
            n_epochs=config.probe_epochs, batch_size=config.probe_batch_size,
            lr=config.probe_lr, seed=config.seed,
        )
        obf_probes[layer_idx] = probe
        obf_metrics[layer_idx] = metrics

    save_probes(obf_probes, obf_metrics, ckpt_obf_probes)
    print(f"Saved retrained probes to {ckpt_obf_probes}")

# Evaluate retrained probes
print("\nEvaluating RETRAINED probes on OBFUSCATED model activations:")
print(f"{'Layer':>6} | {'AUROC':>7} | {'TPR@1%FPR':>10} | {'vs Base':>8}")
print("-" * 44)

obf_eval_retrained = {}
for l in probe_layers:
    probe = obf_probes[l]
    probe.eval()
    with torch.no_grad():
        scores = probe.predict(obf_mean_acts[l][test_idx].to(device)).cpu().numpy()
    result = compute_tpr_at_fpr(scores, test_labels)
    obf_eval_retrained[l] = result
    delta = result["tpr_at_fpr"] - base_eval[l]["tpr_at_fpr"]
    print(f"{l:>6} | {result['auroc']:>7.4f} | {result['tpr_at_fpr']:>10.4f} | {delta:>+8.4f}")

### 4.4 Summary Visualization

The final comparison showing the paper's core result: RL obfuscation degrades linear probe detection.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

layers = probe_layers

# --- Top left: AUROC comparison across extracted layers ---
ax = axes[0, 0]
w = 0.25
x = np.arange(len(layers))
ax.bar(x - w, [base_eval[l]["auroc"] for l in layers], w, label="Base model", color="steelblue", alpha=0.8)
ax.bar(x, [obf_eval_original_probe[l]["auroc"] for l in layers], w, label="Obf (original probe)", color="coral", alpha=0.8)
ax.bar(x + w, [obf_eval_retrained[l]["auroc"] for l in layers], w, label="Obf (retrained probe)", color="mediumseagreen", alpha=0.8)
ax.set_xlabel("Layer"); ax.set_ylabel("AUROC")
ax.set_title("AUROC by Layer")
ax.set_xticks(x); ax.set_xticklabels(layers)
ax.legend(fontsize=8); ax.set_ylim(0.3, 1.05)

# --- Top right: TPR@1%FPR comparison ---
ax = axes[0, 1]
ax.bar(x - w, [base_eval[l]["tpr_at_fpr"] for l in layers], w, label="Base model", color="steelblue", alpha=0.8)
ax.bar(x, [obf_eval_original_probe[l]["tpr_at_fpr"] for l in layers], w, label="Obf (original probe)", color="coral", alpha=0.8)
ax.bar(x + w, [obf_eval_retrained[l]["tpr_at_fpr"] for l in layers], w, label="Obf (retrained probe)", color="mediumseagreen", alpha=0.8)
ax.set_xlabel("Layer"); ax.set_ylabel("TPR at 1% FPR")
ax.set_title("Detection Rate by Layer (at 1% FPR)")
ax.set_xticks(x); ax.set_xticklabels(layers)
ax.legend(fontsize=8); ax.set_ylim(0, 1.05)

# --- Bottom left: ROC curves for best layer ---
ax = axes[1, 0]
r = base_eval[best_layer]
ax.plot(r["fpr_curve"], r["tpr_curve"], label=f"Base (AUC={r['auroc']:.3f})", color="steelblue", linewidth=2)
r = obf_eval_original_probe[best_layer]
ax.plot(r["fpr_curve"], r["tpr_curve"], label=f"Obf, orig probe (AUC={r['auroc']:.3f})", color="coral", linewidth=2)
r = obf_eval_retrained[best_layer]
ax.plot(r["fpr_curve"], r["tpr_curve"], label=f"Obf, retrained (AUC={r['auroc']:.3f})", color="mediumseagreen", linewidth=2)
ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title(f"ROC Curves — Layer {best_layer}")
ax.legend(fontsize=9)
ax.axvline(x=0.01, color="red", linestyle=":", alpha=0.5, label="1% FPR")

# --- Bottom right: Summary table as text ---
ax = axes[1, 1]
ax.axis("off")
summary_text = f"""Summary Results (Best Layer: {best_layer})
{'='*45}

                     AUROC    TPR@1%FPR
                     -----    ---------
Base model:          {base_eval[best_layer]['auroc']:.4f}    {base_eval[best_layer]['tpr_at_fpr']:.4f}
Obf (orig probe):    {obf_eval_original_probe[best_layer]['auroc']:.4f}    {obf_eval_original_probe[best_layer]['tpr_at_fpr']:.4f}
Obf (retrained):     {obf_eval_retrained[best_layer]['auroc']:.4f}    {obf_eval_retrained[best_layer]['tpr_at_fpr']:.4f}

Paper (Llama 2 7B, Layer 10, mean agg):
  Base model TPR@1%FPR:    ~0.96
  Obf (orig probe):       ~0.59 evasion
  Obf (retrained):        partially recovered

Layers extracted: {probe_layers[0]}..{probe_layers[-1]} ({len(probe_layers)}/{n_layers})

Evasion rate (ours):
  Original probe: {max(0, 1 - obf_eval_original_probe[best_layer]['tpr_at_fpr'] / max(base_eval[best_layer]['tpr_at_fpr'], 0.01)):.1%}
  Retrained probe: {max(0, 1 - obf_eval_retrained[best_layer]['tpr_at_fpr'] / max(base_eval[best_layer]['tpr_at_fpr'], 0.01)):.1%}
"""
ax.text(0.05, 0.95, summary_text, transform=ax.transAxes, fontsize=11,
        verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.suptitle("RL-Obfuscation: Linear Probe Evasion Results", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("obfuscation_results.png", dpi=150, bbox_inches="tight")
plt.show()

---
# Conclusion

We reproduced the core linear probe experiment from the RL-Obfuscation paper:

1. **Baseline probes work well:** Linear probes on mid-to-late layers achieve high AUROC and TPR at 1% FPR, confirming that harmful behavior leaves clear signatures in the residual stream.

2. **RL obfuscation degrades detection:** After GRPO training with probe-based rewards, the model's activations change enough to fool the original probe. The TPR at 1% FPR drops substantially.

3. **Retrained probes partially recover:** Training new probes on the obfuscated model's activations can recover some detection capability, but typically not to the original level.

**Limitations of this reproduction:**
- We used a smaller model (1.5B vs 7-14B) which may show different evasion dynamics
- Our blackbox reward uses keyword matching instead of the paper's StrongReject Gemma-2B classifier
- We trained on fewer prompts (200 vs 1000) for compute efficiency
- We did not implement curriculum training (progressively tightening the threshold)

**Next steps for full reproduction:**
- Scale to Llama 2 7B or Llama 3 8B with the paper's exact datasets
- Implement StrongReject reward model (Gemma-2B + PEFT adapter)
- Add curriculum training (`rl_utils/curriculum_trainer.py`)
- Compare aggregation methods (mean vs max) — the paper's key finding is that max-pooling probes resist evasion
- Test MLP and attention-based probe architectures